In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName ("ReduceByKey_vs_GroupByKey") \
.getOrCreate ()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/08 22:47:29 INFO SparkEnv: Registering MapOutputTracker
26/04/08 22:47:29 INFO SparkEnv: Registering BlockManagerMaster
26/04/08 22:47:29 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/08 22:47:29 INFO SparkEnv: Registering OutputCommitCoordinator


In [2]:
!hadoop fs -ls /tmp/

Found 7 items
-rw-r--r--   2 sowapipython hadoop   10528211 2026-04-01 18:47 /tmp/customers_10mb.csv
-rw-r--r--   2 sowapipython hadoop  168541068 2026-04-01 18:48 /tmp/customers_150mb.csv
-rw-r--r--   2 sowapipython hadoop    1060750 2026-04-01 18:47 /tmp/customers_1mb.csv
-rw-r--r--   2 sowapipython hadoop  343317147 2026-04-01 18:48 /tmp/customers_300mb.csv
drwxrwxrwt   - hdfs         hadoop          0 2026-03-31 14:46 /tmp/hadoop-yarn
drwx-wx-wx   - hive         hadoop          0 2026-03-31 14:47 /tmp/hive
-rw-r--r--   2 root         hadoop         84 2026-04-01 17:13 /tmp/inputhdfsdbz.txt


In [4]:
# Using 1mb file
hdfs_path = '/tmp/customers_1mb.csv'
rdd = spark.sparkContext.textFile(hdfs_path)

In [5]:
header = rdd.first()

In [6]:
rdd_no_header = rdd.filter(lambda row:row!= header).map(lambda row:row.split(','))

In [7]:
rdd_no_header.first()

['0', 'Customer_0', 'Pune', 'Maharashtra', 'India', '2023-06-29', 'False']

In [ ]:
#### REDUCEBYKEY

In [10]:
reduced_by_rdd = rdd_no_header.map(lambda row:(row[2],1)).reduceByKey(lambda x,y: x+y)

In [11]:
reduced_by_rdd.collect()

[('Pune', 2243),
 ('Hyderabad', 2242),
 ('Mumbai', 2142),
 ('Delhi', 2200),
 ('Bangalore', 2211),
 ('Ahmedabad', 2198),
 ('Chennai', 2194),
 ('Kolkata', 2223)]

In [ ]:
#### GROUPBYKEY

In [13]:
grouped_by_rddd = rdd_no_header.map(lambda row:(row[2],1)).groupByKey()

In [14]:
grouped_by_rddd.collect()

[('Pune', <pyspark.resultiterable.ResultIterable at 0x7fdd3df1dcd0>),
 ('Hyderabad', <pyspark.resultiterable.ResultIterable at 0x7fdd3df139d0>),
 ('Mumbai', <pyspark.resultiterable.ResultIterable at 0x7fdd3d5f3450>),
 ('Delhi', <pyspark.resultiterable.ResultIterable at 0x7fdd3dee37d0>),
 ('Bangalore', <pyspark.resultiterable.ResultIterable at 0x7fdd3d4b6f50>),
 ('Ahmedabad', <pyspark.resultiterable.ResultIterable at 0x7fdd3d4b73d0>),
 ('Chennai', <pyspark.resultiterable.ResultIterable at 0x7fdd3d4b5550>),
 ('Kolkata', <pyspark.resultiterable.ResultIterable at 0x7fdd3d4b6990>)]

In [16]:
grouped_by_result = grouped_by_rddd.map(lambda row: (row[0], len(row[1])))

In [17]:
grouped_by_result.collect()

[('Pune', 2243),
 ('Hyderabad', 2242),
 ('Mumbai', 2142),
 ('Delhi', 2200),
 ('Bangalore', 2211),
 ('Ahmedabad', 2198),
 ('Chennai', 2194),
 ('Kolkata', 2223)]

In [24]:
# Using 300mb file
rdd = spark.sparkContext.textFile("/tmp/customers_300mb.csv")

In [25]:
## Reduce By Key in 1 shot
rdd.map(lambda row: row.split(',')).map(lambda row: (row[2], 1)).reduceByKey(lambda x, y: x + y).collect()

[('Mumbai', 661241),
 ('Kolkata', 660174),
 ('Hyderabad', 662281),
 ('Bangalore', 661013),
 ('Chennai', 660249),
 ('Ahmedabad', 660218),
 ('city', 1),
 ('Pune', 660737),
 ('Delhi', 661025)]

In [30]:
rdd.map(lambda row: row.split(',')) .map(lambda row: (row[2], 1)).groupByKey() .map(lambda row: (row[0], len(list(row[1])))).collect()

[('Mumbai', 661241),
 ('Kolkata', 660174),
 ('Hyderabad', 662281),
 ('Bangalore', 661013),
 ('Chennai', 660249),
 ('Ahmedabad', 660218),
 ('Delhi', 661025),
 ('Pune', 660737),
 ('city', 1)]

In [31]:
spark.stop()